In [ ]:
import pvlib
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import os


In [ ]:
### Location
lat = 55.6 #degrees North
lon = 12.1 #degrees East
albedo = 0.2

### Dates
start = pd.Timestamp('2023-01-01 00:00:00', tz='Europe/Copenhagen')
end = pd.Timestamp('2023-12-31 23:59:00', tz='Europe/Copenhagen')

### Module parameters
module_width = 1
module_length = 1.6

### System parameters
pitch = 11.75 #estimated from other measurements
length = 3.3 #estimated from other measurements
hub_height = 1.95
gcr = 0.28
max_angle = 60
axis_tilt = 0
axis_azimuth = 180

skymodel = 'haydavies' #options: perez, perez-driesse, haydavies, isotropic

In [ ]:
# create location variable
location = pvlib.location.Location(latitude=lat, longitude=lon)
if os.path.isfile(f"data/cams/cams_radiation_{lat}_{lon}_{start}_{end}.csv"):
    data = pd.read_csv(f'data/cams/cams_radiation_{lat}_{lon}_{start}_{end}.csv', index_col=0, parse_dates=True)
else:
    data, meta = pvlib.iotools.get_cams(lat, lon,start,end,'arule@dtu.dk', identifier='cams_radiation',time_step='1min') 
    data.to_csv(f'data/cams/cams_radiation_{lat}_{lon}_{start}_{end}.csv')

In [ ]:
# get solar position and module tilt angle
solar_position = location.get_solarposition(data.index)

#get reference module tilt and POA
module_tilt_ref = pvlib.tracking.singleaxis(solar_position['zenith'], solar_position['azimuth'],backtrack=True, gcr=gcr, max_angle=max_angle,axis_azimuth=180)
surface_orientation_ref = pvlib.tracking.calc_surface_orientation(module_tilt_ref['tracker_theta'], axis_tilt = axis_tilt, axis_azimuth=axis_azimuth)

dni_extra = pvlib.irradiance.get_extra_radiation(solar_position['zenith'])

if skymodel == 'perez-driesse':
    poa_sky_diffuse = pvlib.irradiance.perez_driesse(module_tilt_ref['surface_tilt'],module_tilt_ref['surface_azimuth'],\
                                                        data['dhi'], data['dni'], dni_extra,\
                                                        solar_position['zenith'], solar_position['azimuth'])
elif skymodel == 'isotropic':
    poa_sky_diffuse = pvlib.irradiance.isotropic(module_tilt_ref['surface_tilt'], data['dhi'])
elif skymodel == 'haydavies':
    poa_sky_diffuse = pvlib.irradiance.haydavies(module_tilt_ref['surface_tilt'], module_tilt_ref['surface_azimuth'],\
                                                    data['dhi'], data['dni'], dni_extra, \
                                                    solar_position['zenith'], solar_position['azimuth'])


#get direct and ground-reflected irradiance contributions
poa_direct = pvlib.irradiance.beam_component(module_tilt_ref['surface_tilt'], module_tilt_ref['surface_azimuth'],\
                                              solar_position['zenith'], solar_position['azimuth'],data['dni'])
poa_ground = pvlib.irradiance.get_ground_diffuse(module_tilt_ref['surface_tilt'], data['ghi'], albedo)

poa_total_ref = poa_sky_diffuse + poa_direct + poa_ground

# module_tilt

In [ ]:
module_theta = np.arange(-max_angle, max_angle+1, 1) #module tilt angles to test


module_theta_2D = np.asarray(module_theta)[None, :]
ghi_2D = np.asarray(data['ghi'])[:, None]
dhi_2D = np.asarray(data['dhi'])[:, None]
dni_2D = np.asarray(data['dni'])[:, None]
solar_zenith_2D = np.asarray(solar_position['zenith'])[:, None]
solar_azimuth_2D = np.asarray(solar_position['azimuth'])[:, None]
dni_extra_2D = pvlib.irradiance.get_extra_radiation(solar_zenith_2D)


In [ ]:
#convert tilt and axis azimuth to surface tilt and surface azimuth
surface_orientation = pvlib.tracking.calc_surface_orientation(module_theta_2D, axis_tilt = axis_tilt, axis_azimuth=axis_azimuth)
surface_tilt_2D = surface_orientation['surface_tilt']
surface_azimuth_2D = surface_orientation['surface_azimuth']

In [ ]:
# run transposition model for each tilt angle at each timestamp
# skymodel = 'isotropic'
if skymodel == 'perez-driesse':
    poa_sky_diffuse = pvlib.irradiance.perez_driesse(surface_tilt_2D,surface_azimuth_2D, dhi_2D, dni_2D, dni_extra_2D, solar_zenith_2D, solar_azimuth_2D)
elif skymodel == 'isotropic':
    poa_sky_diffuse = pvlib.irradiance.isotropic(surface_tilt_2D, dhi_2D)
elif skymodel == 'haydavies':
    poa_sky_diffuse = pvlib.irradiance.haydavies(surface_tilt_2D, surface_azimuth_2D,dhi_2D, dni_2D, dni_extra_2D,solar_zenith_2D, solar_azimuth_2D)

#get direct and ground-reflected irradiance contributions
poa_direct = pvlib.irradiance.beam_component(surface_tilt_2D, surface_azimuth_2D, solar_zenith_2D, solar_azimuth_2D,dni_2D)
poa_ground = pvlib.irradiance.get_ground_diffuse(surface_tilt_2D, ghi_2D, albedo)

poa_total = poa_sky_diffuse + poa_direct + poa_ground

# get optimal tilt angle for each time step
optimal_tilt_indices = np.argmax(poa_total, axis=1)
optimal_tilt = np.asarray(module_theta)[optimal_tilt_indices]
row_indices = np.arange(poa_total.shape[0])
poa_optimal = np.asarray(poa_total)[row_indices, optimal_tilt_indices]

In [ ]:
poa_gain_instantaneous = 100*(poa_optimal - poa_total_ref) / poa_total_ref

annual_poa_optimal = poa_optimal.sum()
annual_poa_reference = poa_total_ref.sum()

poa_gain_percent = (annual_poa_optimal - annual_poa_reference)/annual_poa_reference*100

poa_gain_percent

In [ ]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

hour_ticks = [f"{hour:02d}:00" for hour in range(0,24,2)]

# Reference tilt from the tracker calculation
reference_tilt = module_tilt_ref["tracker_theta"]

# Convert to a NumPy array while preserving the existing data order
if isinstance(reference_tilt, pd.Series):
    reference_tilt = reference_tilt.to_numpy()
else:
    reference_tilt = np.asarray(reference_tilt)

if len(reference_tilt) != len(data.index):
    raise ValueError(
        "module_tilt_setting['tracker_theta'] and data must have the same length."
    )

# Convert timestamps to local Copenhagen time
plot_index = data.index.tz_convert("Europe/Copenhagen")

dates_to_plot = [
    ("2023-05-23", "May 23, 2023"),
    ("2023-06-07", "June 7, 2023"),
]

fig = make_subplots(
    rows=1,
    cols=2,
    specs=[
        [
            {"secondary_y": True},
            {"secondary_y": True},
        ]
    ],
    subplot_titles=[title for _, title in dates_to_plot],
    horizontal_spacing=0.2,
)

for col, (date_string, title) in enumerate(dates_to_plot, start=1):
    day = pd.Timestamp(date_string, tz="Europe/Copenhagen")
    next_day = day + pd.Timedelta(days=1)

    day_mask = (plot_index >= day) & (plot_index < next_day)

    if not day_mask.any():
        raise ValueError(f"No data found for {date_string}.")

    # Use time of day on both x-axes so the two days are directly comparable
    time_of_day = plot_index[day_mask].strftime("%H:%M")

    # POA at the optimal tilt
    fig.add_trace(
        go.Scatter(
            x=time_of_day,
            y=np.asarray(poa_optimal)[day_mask],
            mode="lines",
            name="POA at optimal tilt",
            line=dict(color="red"),
            legendgroup="poa",
            showlegend=(col == 1),
        ),
        row=1,
        col=col,
        secondary_y=True,
    )

    # Best tilt among the tested candidate angles
    fig.add_trace(
        go.Scatter(
            x=time_of_day,
            y=np.asarray(optimal_tilt)[day_mask],
            mode="lines",
            name="Optimal tilt",
            line=dict(color="green"),
            legendgroup="optimal_tilt",
            showlegend=(col == 1),
        ),
        row=1,
        col=col,
        secondary_y=False,
    )

    # Reference/tracker tilt
    fig.add_trace(
        go.Scatter(
            x=time_of_day,
            y=reference_tilt[day_mask],
            mode="lines",
            name="Reference tilt",
            line=dict(color="black"),
            legendgroup="reference_tilt",
            showlegend=(col == 1),
        ),
        row=1,
        col=col,
        secondary_y=False,
    )

    fig.update_xaxes(
        title_text="Time",
        tickmode="array",
        tickvals=hour_ticks,
        ticktext=hour_ticks,
        tickangle=-45,
        row=1,
        col=col,
    )

    fig.update_yaxes(
        title_text="Tilt (degrees)",
        row=1,
        col=col,
        secondary_y=False,
    )

    fig.update_yaxes(
        title_text="Irradiance (W/m²)",
        row=1,
        col=col,
        secondary_y=True,
    )

fig.update_layout(
    title="Plane-of-array irradiance and tilt comparison",
    template="plotly_white",
    hovermode="x unified",
    autosize=True,
    height=500,

    legend=dict(
        orientation="h",
        yanchor="bottom",
        y=1.1,
        xanchor="right",
        x=1,
    ),

    margin=dict(
        l=70,
        r=30,
        t=100,
        b=80,
    ),
)

fig.show()

In [ ]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Reference tilt from the tracker calculation
reference_tilt = module_tilt_ref["tracker_theta"]

if isinstance(reference_tilt, pd.Series):
    reference_tilt = reference_tilt.to_numpy()
else:
    reference_tilt = np.asarray(reference_tilt)

poa_optimal = np.asarray(poa_optimal).squeeze()
poa_total_ref = np.asarray(poa_total_ref).squeeze()
poa_gain_instantaneous = np.asarray(poa_gain_instantaneous).squeeze()
optimal_tilt = np.asarray(optimal_tilt).squeeze()

n_points = len(data.index)

for name, values in {
    "reference tilt": reference_tilt,
    "poa_optimal": poa_optimal,
    "poa_total_ref": poa_total_ref,
    "poa_gain_instantaneous": poa_gain_instantaneous,
    "optimal_tilt": optimal_tilt,
}.items():
    if len(values) != n_points:
        raise ValueError(
            f"{name} has length {len(values)}, but data has length {n_points}."
        )

# Use local Copenhagen timestamps
plot_index = data.index.tz_convert("Europe/Copenhagen")

fig = make_subplots(
    rows=1,
    cols=1,
    specs=[[{"secondary_y": True}]],
)

# Reference POA: add this first so the optimal POA can fill to it
fig.add_trace(
    go.Scatter(
        x=plot_index,
        y=poa_total_ref,
        mode="lines",
        name="POA reference",
        line=dict(color="royalblue", width=1),
        legendgroup="poa",
    ),
    secondary_y=True,
)

# POA at optimal tilt, with fill to the preceding reference POA trace
fig.add_trace(
    go.Scatter(
        x=plot_index,
        y=poa_optimal,
        mode="lines",
        name="POA optimal",
        line=dict(color="red", width=1),
        fill="tonexty",
        fillcolor="rgba(0, 180, 0, 0.20)",
        legendgroup="poa",
    ),
    secondary_y=True,
)

# Optimal tilt
fig.add_trace(
    go.Scatter(
        x=plot_index,
        y=optimal_tilt,
        mode="lines",
        name="Optimal tilt",
        line=dict(color="green", width=1),
        legendgroup="tilt",
    ),
    secondary_y=False,
)

# Reference/tracker tilt
fig.add_trace(
    go.Scatter(
        x=plot_index,
        y=reference_tilt,
        mode="lines",
        name="Reference tilt",
        line=dict(color="black", width=1),
        legendgroup="tilt",
    ),
    secondary_y=False,
)

# Third y-axis: instantaneous POA gain
fig.add_trace(
    go.Scatter(
        x=plot_index,
        y=poa_gain_instantaneous,
        mode="lines",
        name="Instantaneous POA gain",
        line=dict(color="purple", width=1),
        yaxis="y3",
    )
)

fig.update_layout(
    title="Plane-of-array irradiance, tilt, and instantaneous gain",
    template="plotly_white",
    hovermode="x unified",
    autosize=True,
    height=550,
    legend=dict(
        orientation="h",
        yanchor="bottom",
        y=1.02,
        xanchor="right",
        x=1,
    ),
    margin=dict(
        l=70,
        r=140,
        t=100,
        b=70,
    ),

    # Third y-axis for instantaneous POA gain
    yaxis3=dict(
        overlaying="y",
        side="right",
        anchor="free",
        position=0.94,
        showgrid=False,
        tickfont=dict(color="purple"),
        title=dict(
            text="Instantaneous POA gain",
            font=dict(color="purple"),
        ),
    ),
)

fig.update_xaxes(
    title_text="Date",
    dtick="M1",
    tickformat="%b",
    tickangle=-45,
)

fig.update_yaxes(
    title_text="Tilt (degrees)",
    secondary_y=False,
)

fig.update_yaxes(
    title_text="Irradiance (W/m²)",
    secondary_y=True,
)

fig.show()